# 📘 W8 Python Lab — Alpaca Paper Trading + Risk Guards

**n8n W8 강의의 Python 버전. 8주 과정의 대미.** AI verdict를 실제 가상 주문으로 체결.

## 학습 목표
- Alpaca Paper Trading API로 실제 매수/매도 주문 체결
- Risk Guards 4단계 (포지션 사이징·손절·익절·일일 손실 한도)
- Bracket Order (매수+익절+손절 원샷)
- W7 Orchestrator verdict → 자동 주문 연결
- 일일 성과 추적 + CSV 로그

## 🛠 사전 준비

```bash
pip install requests pandas python-dotenv
```

`.env` 추가:
```
ALPACA_API_KEY_ID=여러분의_Paper_Key_ID
ALPACA_API_SECRET=여러분의_Paper_Secret
```

> 🚨 **반드시 Paper URL 사용.** `paper-api.alpaca.markets` (live가 아님!)

## 1. 환경 로드 + 기본 설정

In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
import requests
import pandas as pd

load_dotenv()

# 🚨 절대 URL 바꾸지 말 것 — paper 아니면 실계좌
ALPACA_BASE_URL = 'https://paper-api.alpaca.markets'

ALPACA_HEADERS = {
    'APCA-API-KEY-ID': os.getenv('ALPACA_API_KEY_ID'),
    'APCA-API-SECRET-KEY': os.getenv('ALPACA_API_SECRET'),
    'Content-Type': 'application/json'
}

# 체크
assert os.getenv('ALPACA_API_KEY_ID'), '❌ ALPACA_API_KEY_ID 미설정'
assert 'paper' in ALPACA_BASE_URL, '❌ Paper URL 아님!'
print(f'✓ Alpaca Paper 연결 준비 완료')
print(f'  URL: {ALPACA_BASE_URL}')

## 2. 계좌 정보 조회 (연결 테스트)

In [ ]:
def get_account() -> dict:
    """계좌 정보 조회 — 연결 테스트용."""
    r = requests.get(f'{ALPACA_BASE_URL}/v2/account', headers=ALPACA_HEADERS, timeout=10)
    r.raise_for_status()
    return r.json()

account = get_account()
print('📊 계좌 정보:')
print(f'  계좌번호: {account["account_number"]}')
print(f'  현재 자본: ${float(account["equity"]):,.2f}')
print(f'  전일 자본: ${float(account["last_equity"]):,.2f}')
print(f'  구매 여력: ${float(account["buying_power"]):,.2f}')
print(f'  현금: ${float(account["cash"]):,.2f}')
print(f'  상태: {account["status"]}')

## 3. Risk Guards — 4단계 안전장치

주문 실행 전 반드시 통과해야 하는 필터.

In [ ]:
# === Guard 1: Position Sizing ===
def calculate_position_size(
    equity: float,
    current_price: float,
    confidence: int = 3,
    base_pct: float = 0.02,
    max_pct: float = 0.05
) -> int:
    """자본의 2~4%를 한 포지션에 할당. confidence로 가중.
    
    Returns: 매수 주식 수 (최소 1주, 최대 자본의 5%)
    """
    multiplier_map = {1: 0.5, 2: 0.75, 3: 1.0, 4: 1.5, 5: 2.0}
    mult = multiplier_map.get(confidence, 1.0)
    
    budget = equity * base_pct * mult
    qty = int(budget // current_price)
    
    # 최대 5% 한도
    max_qty = int((equity * max_pct) // current_price)
    
    return max(1, min(qty, max_qty))

# 테스트
qty_conf3 = calculate_position_size(equity=100000, current_price=180, confidence=3)
qty_conf5 = calculate_position_size(equity=100000, current_price=180, confidence=5)
print(f'자본 $100K, 주가 $180:')
print(f'  Confidence 3 → {qty_conf3}주 (${qty_conf3 * 180}, {qty_conf3 * 180 / 1000:.1f}%)')
print(f'  Confidence 5 → {qty_conf5}주 (${qty_conf5 * 180}, {qty_conf5 * 180 / 1000:.1f}%)')

In [ ]:
# === Guard 2&3: Stop Loss + Take Profit 가격 계산 ===
def calculate_stops(
    entry_price: float,
    stop_loss_pct: float = 0.05,
    take_profit_pct: float = 0.10
) -> dict:
    """손절가 / 익절가 계산."""
    return {
        'stop_loss': round(entry_price * (1 - stop_loss_pct), 2),
        'take_profit': round(entry_price * (1 + take_profit_pct), 2),
        'stop_limit': round(entry_price * (1 - stop_loss_pct) * 0.995, 2)  # slippage 보호
    }

stops = calculate_stops(entry_price=180)
print('📐 $180 매수 시 자동 주문 가격:')
print(f'  Take Profit (+10%): ${stops["take_profit"]}')
print(f'  Stop Loss (-5%):    ${stops["stop_loss"]}')
print(f'  Stop Limit:         ${stops["stop_limit"]}')

In [ ]:
# === Guard 4: Daily Loss Limit ===
def check_daily_loss_guard(
    equity: float,
    last_equity: float,
    max_daily_loss_pct: float = -2.0
) -> dict:
    """당일 손실이 한도를 넘었는지 체크."""
    daily_pnl_pct = ((equity - last_equity) / last_equity) * 100
    
    if daily_pnl_pct < max_daily_loss_pct:
        return {
            'allowed': False,
            'reason': f'일일 손실 한도 초과: {daily_pnl_pct:.2f}%',
            'daily_pnl_pct': round(daily_pnl_pct, 2)
        }
    
    return {
        'allowed': True,
        'daily_pnl_pct': round(daily_pnl_pct, 2)
    }

guard = check_daily_loss_guard(
    equity=float(account['equity']),
    last_equity=float(account['last_equity'])
)
print(f'🛡️ 일일 손실 Guard: {"✓ 통과" if guard["allowed"] else "🚫 차단"}')
print(f'   당일 수익률: {guard["daily_pnl_pct"]}%')

## 4. Bracket Order — 매수+익절+손절 원샷

Alpaca의 강력한 기능. 매수와 동시에 자동 청산 로직 내장.

In [ ]:
def place_bracket_buy(
    symbol: str,
    qty: int,
    entry_price: float,
    stop_loss_pct: float = 0.05,
    take_profit_pct: float = 0.10
) -> dict:
    """Bracket Order — 매수 + 자동 익절 + 자동 손절."""
    stops = calculate_stops(entry_price, stop_loss_pct, take_profit_pct)
    
    payload = {
        'symbol': symbol,
        'qty': str(qty),
        'side': 'buy',
        'type': 'market',
        'time_in_force': 'day',
        'order_class': 'bracket',
        'take_profit': {
            'limit_price': str(stops['take_profit'])
        },
        'stop_loss': {
            'stop_price': str(stops['stop_loss']),
            'limit_price': str(stops['stop_limit'])
        }
    }
    
    r = requests.post(
        f'{ALPACA_BASE_URL}/v2/orders',
        headers=ALPACA_HEADERS,
        json=payload,
        timeout=10
    )
    
    if r.status_code in [200, 201]:
        order = r.json()
        return {
            'success': True,
            'order_id': order['id'],
            'symbol': symbol,
            'qty': qty,
            'entry_price': entry_price,
            'stop_loss': stops['stop_loss'],
            'take_profit': stops['take_profit'],
            'status': order['status']
        }
    else:
        return {
            'success': False,
            'error': r.text,
            'status_code': r.status_code
        }

# 예시: SPY(S&P 500 ETF) 1주 매수 (테스트용)
# ⚠ 실행하면 실제로 모의 주문됨
# test_order = place_bracket_buy('SPY', qty=1, entry_price=580)
# print(json.dumps(test_order, indent=2))
print('⚠ 실제 주문은 아래 통합 파이프라인 셀에서 실행됩니다')

## 5. 현재가 조회 (Alpaca 시세 API)

In [ ]:
def get_latest_price(symbol: str) -> float:
    """Alpaca의 최신 시세 조회."""
    url = f'https://data.alpaca.markets/v2/stocks/{symbol}/trades/latest'
    r = requests.get(url, headers=ALPACA_HEADERS, timeout=10)
    r.raise_for_status()
    return float(r.json()['trade']['p'])

# 테스트
price = get_latest_price('AAPL')
print(f'AAPL 현재가: ${price}')

## 6. verdict → 주문 자동 변환 파이프라인

W7 Orchestrator가 낸 verdict를 받아 주문까지 연결.

In [ ]:
def execute_verdict(verdict: dict, dry_run: bool = True) -> dict:
    """verdict를 받아 Risk Guards 통과 시 주문 실행.
    
    Args:
        verdict: {'ticker', 'action' (BUY/SELL/HOLD), 'confidence' (1~5)}
        dry_run: True면 시뮬레이션만, False면 실제 주문
    """
    ticker = verdict['ticker']
    action = verdict.get('action', 'HOLD')
    confidence = verdict.get('confidence', 3)
    
    result = {
        'ticker': ticker,
        'action': action,
        'confidence': confidence,
        'timestamp': datetime.now().isoformat(),
        'guards': {},
        'executed': False
    }
    
    # 체크 1: BUY & confidence >= 4만 통과
    if action != 'BUY' or confidence < 4:
        result['skip_reason'] = f'{action} conf={confidence} — 기준 미달 (BUY & conf≥4 필요)'
        return result
    
    # 체크 2: Daily Loss Guard
    acc = get_account()
    daily_check = check_daily_loss_guard(
        equity=float(acc['equity']),
        last_equity=float(acc['last_equity'])
    )
    result['guards']['daily_loss'] = daily_check
    
    if not daily_check['allowed']:
        result['skip_reason'] = daily_check['reason']
        return result
    
    # 체크 3: 현재가 조회 + 포지션 사이즈 계산
    try:
        current_price = get_latest_price(ticker)
    except Exception as e:
        result['skip_reason'] = f'가격 조회 실패: {e}'
        return result
    
    qty = calculate_position_size(
        equity=float(acc['equity']),
        current_price=current_price,
        confidence=confidence
    )
    
    stops = calculate_stops(current_price)
    
    result['plan'] = {
        'symbol': ticker,
        'qty': qty,
        'entry_price': current_price,
        'stop_loss': stops['stop_loss'],
        'take_profit': stops['take_profit'],
        'budget_used': round(qty * current_price, 2),
        'equity_pct': round((qty * current_price) / float(acc['equity']) * 100, 2)
    }
    
    # 체크 4: 실제 주문 (dry_run이면 시뮬만)
    if dry_run:
        result['executed'] = False
        result['note'] = 'DRY RUN — 실제 주문 안 됨'
    else:
        order = place_bracket_buy(ticker, qty, current_price)
        result['executed'] = order.get('success', False)
        result['order'] = order
    
    return result

# 테스트 — DRY RUN
test_verdict = {
    'ticker': 'AAPL',
    'action': 'BUY',
    'confidence': 4
}

result = execute_verdict(test_verdict, dry_run=True)
print(json.dumps(result, indent=2, ensure_ascii=False))

## 7. 실제 주문 실행 (원할 때만)

⚠ 주의: 아래 셀 실행 시 실제로 Paper 계좌에 주문됩니다 (가상 자본이지만).

In [ ]:
# 진짜 주문 실행 (주석 해제 후 실행)
# 
# 📌 미국 장 시간 외에는 주문이 'pending'으로 대기하다가
#     다음 개장에 체결됩니다 (time_in_force='day' 기준)

# live_result = execute_verdict(test_verdict, dry_run=False)
# print(json.dumps(live_result, indent=2, ensure_ascii=False))

print('⚠ 실제 주문을 실행하려면 위 코드 주석 해제')
print('   현재 dry_run=True로 시뮬레이션만 실행됨')

## 8. 포지션 및 주문 조회

In [ ]:
def get_positions() -> list[dict]:
    """현재 보유 포지션."""
    r = requests.get(f'{ALPACA_BASE_URL}/v2/positions', headers=ALPACA_HEADERS, timeout=10)
    return r.json() if r.ok else []

def get_orders(status: str = 'all', limit: int = 50) -> list[dict]:
    """주문 목록. status: open / closed / all."""
    r = requests.get(
        f'{ALPACA_BASE_URL}/v2/orders',
        headers=ALPACA_HEADERS,
        params={'status': status, 'limit': limit},
        timeout=10
    )
    return r.json() if r.ok else []

# 현재 포지션
positions = get_positions()
if positions:
    print(f'📊 현재 포지션 ({len(positions)}개):')
    for p in positions:
        pnl_pct = float(p['unrealized_plpc']) * 100
        print(f"  {p['symbol']}: {p['qty']}주 @ ${float(p['avg_entry_price']):.2f} "
              f"→ ${float(p['current_price']):.2f} ({pnl_pct:+.2f}%)")
else:
    print('📊 현재 보유 포지션 없음')

# 최근 주문
orders = get_orders(status='all', limit=5)
if orders:
    print(f'\n📋 최근 주문 {len(orders)}건:')
    for o in orders[:5]:
        print(f"  [{o['status']}] {o['side'].upper()} {o['symbol']} x{o['qty']} "
              f"@ {o.get('filled_avg_price', 'pending')}")
else:
    print('\n📋 주문 이력 없음')

## 9. 일일 성과 추적 + CSV 로그

n8n Sheets 기록 = pandas CSV.

In [ ]:
def log_daily_performance(csv_path: str = 'performance_log.csv'):
    """매일 저녁 실행 — 당일 성과를 CSV에 append."""
    acc = get_account()
    positions = get_positions()
    orders = get_orders(status='closed', limit=50)
    
    today = datetime.now().strftime('%Y-%m-%d')
    
    # 당일 체결 주문만
    today_orders = [o for o in orders if o.get('filled_at', '').startswith(today)]
    
    equity = float(acc['equity'])
    last_equity = float(acc['last_equity'])
    daily_return = (equity - last_equity) / last_equity * 100
    
    row = {
        'date': today,
        'equity': round(equity, 2),
        'daily_return_pct': round(daily_return, 3),
        'cash': round(float(acc['cash']), 2),
        'positions_count': len(positions),
        'orders_today': len(today_orders),
        'buys_today': len([o for o in today_orders if o['side'] == 'buy']),
        'sells_today': len([o for o in today_orders if o['side'] == 'sell'])
    }
    
    path = Path(csv_path)
    df_new = pd.DataFrame([row])
    
    if path.exists():
        df_new.to_csv(path, mode='a', header=False, index=False)
    else:
        df_new.to_csv(path, index=False)
    
    print(f'✓ 일일 성과 기록 완료: {csv_path}')
    print(f'  당일 수익률: {daily_return:+.3f}%')
    print(f'  현재 자본: ${equity:,.2f}')
    print(f'  당일 매매: {len(today_orders)}건 (매수 {row["buys_today"]} / 매도 {row["sells_today"]})')
    
    return row

# 실행
today_performance = log_daily_performance()

## 10. 누적 성과 시각화

며칠 데이터가 쌓이면 실행 가능.

In [ ]:
import matplotlib.pyplot as plt

def visualize_performance(csv_path: str = 'performance_log.csv'):
    """누적 수익률 + 일별 수익률 2패널 차트."""
    if not Path(csv_path).exists():
        print('로그 파일이 없습니다. 먼저 며칠 운영해보세요.')
        return
    
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    
    initial = df['equity'].iloc[0] if len(df) > 1 else 100000
    df['cumulative_return'] = (df['equity'] - initial) / initial * 100
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    
    # 누적 수익률
    ax1.plot(df['date'], df['cumulative_return'], 'o-', color='#7c3aed', linewidth=2)
    ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_ylabel('Cumulative Return (%)')
    ax1.set_title('Paper Trading Performance')
    ax1.grid(alpha=0.3)
    
    # 일별 수익률
    colors = ['#22c55e' if r >= 0 else '#ef4444' for r in df['daily_return_pct']]
    ax2.bar(df['date'], df['daily_return_pct'], color=colors, alpha=0.7)
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.set_ylabel('Daily Return (%)')
    ax2.set_xlabel('Date')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('performance_chart.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('✓ 차트 저장: performance_chart.png')

visualize_performance()

## 🎉 8주 여정 완료!

### 여러분이 만든 것

- ✅ W1~W8 Python 전체 파이프라인 (가격 수집 → 지표 → 뉴스 → 차트 → RAG → 멀티 에이전트 → 자동매매)
- ✅ Alpaca Paper Trading 연동 + Risk Guards 4단계
- ✅ Bracket Order로 자동 익절/손절
- ✅ CSV 기반 성과 추적 + 시각화

### 🎯 최종 과제

1. **1주일 운영** — 매일 저녁 `log_daily_performance()` 실행 → 일주일 후 `visualize_performance()` 차트 생성
2. **W7 통합** — W7의 `orchestrator()` 결과를 파싱해 `execute_verdict()`에 연결 (자동화 완성)
3. **Backtest** — yfinance로 지난 3개월 historical data 받아 에이전트 판단 재현 → 승률 측정
4. **GitHub 레포지토리 생성** — 모든 W1~W8 노트북을 한 레포에 정리 + README 작성
5. **이력서 한 줄 작성**:
   > "Python + Claude API로 멀티 에이전트 투자 리서치 시스템 구축 — Alpaca Paper Trading으로 1주일 승률 XX% 달성, Risk Guards로 최대 드로다운 -XX% 이내 유지"

## 🔄 n8n vs Python 최종 비교

| 개념 | n8n | Python |
|---|---|---|
| Alpaca 주문 | HTTP Request + Header Auth | `requests.post(headers=...)` |
| Bracket Order | JSON body | 동일 JSON body |
| Risk Guards | Code + IF 노드 | Python 함수 + 조건문 |
| verdict 파이프라인 | 시각적 연결 | 함수 호출 chain |
| 성과 저장 | Google Sheets | pandas CSV |
| 시각화 | 외부 (Looker Studio 등) | matplotlib 인라인 |

### 두 방식의 최종 결론

- **n8n**: 시각적 워크플로, 디버깅 쉬움, 비개발자 친화적
- **Python**: 세밀한 제어, 복잡한 로직, 데이터 분석 용이
- **실전에서는 둘 다 씁니다.** 간단한 자동화는 n8n, 복잡한 분석/백테스트는 Python. 서로 HTTP/Webhook으로 연결.

**여러분은 이제 두 도구의 원리를 모두 이해하는 풀스택 AI 엔지니어입니다.** 축하합니다! 🎓